# EA722 – Laboratório de Controle e Servomecanismo

## Experiência 3: Controle PID

Universidade Estadual de Campinas – UNICAMP <br>
Faculdade de Engenharia Elétrica e de Computação – FEEC <br>

**Professores:** Fernando J. Von Zuben / Caíque Santos Lima <br>
**Grupo / Bancada:** T1, T2, R1, R2 ou E <br>
**Turma:** K, L, U ou V <br>
**Aluno(a): Mariana Leister Gonçalves** , **RA:** 233115 <br>
**Aluno(a): Mariana Vasconcelos Silva** , **RA:** 251295 <br>
**Aluno(a): Marina Alves Farias** , **RA:** 188521 <br>

## Bibliotecas

In [25]:
try:
    import control
except ImportError:
    !pip install control
    import control

In [26]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import random
from tqdm import tqdm

# # Google Drive Mount (Colab)
# from google.colab import drive
# drive.mount('/content/drive')

In [27]:
random.seed(42)

## (1) Controle PID por meta-heurística computacional


Nesta parte deve-se empregar uma versão de algoritmo evolutivo (técnica de inteligência computacional), com operadores genéticos dedicados à otimização em espaço contínuos, sendo que se supõe que os parâmetros $k_P$, $k_I$ e $k_D$ excursionam no intervalo $[0,+20]$ e que a planta é:

$$
G(s) = \frac{10}{(s+1)(s+2)(s+3)(s+4)} = \frac{10}{s^4+10s^3+35s^2+50s+24}
$$


Será realizada, portanto, uma busca no espaço $[0,+20]$ × $[0,+20]$ × $[0,+20]$, visando maximizar a função de fitness. Os critérios a serem atendidos inicialmente são:

1.   Tempo de acomodação menor ou igual a 3,0 segundos;
2.   Tempo de subida menor ou igual a 0,6 segundos;
3.   Margem de fase de 50 graus.

A função de *fitness* é tal que penaliza a violação das restrições na forma:

$$
fitness(k_P,k_I,k_D) = \frac{1}{p_1+\frac{3,0}{0,6}p_2+p_3+1},
$$

tendo valor máximo em 1, quando todas as restrições forem atendidas. Cada termo de violação de restrição é dado por:

- O termo $p_1$ é definido em função do **tempo de acomodação** $(t_s)$:

$$
p_1 =
\begin{cases}
0, & \text{se } t_s \leq 3{,}0 \text{ s} \\
t_s - 3{,}0, & \text{se } t_s > 3{,}0 \text{ s}
\end{cases}
$$

- O termo $p_2$ é definido em função do **tempo de subida** $(t_r)$:

$$
p_2 =
\begin{cases}
0, & \text{se } t_r \leq 0{,}6 \text{ s} \\
t_r - 0{,}6, & \text{se } t_r > 0{,}6 \text{ s}
\end{cases}
$$

- O termo $p_3$ depende da **margem de fase** $(MF)$ do sistema de controle:

$$
p_3 = (MF - 50)^2
$$



### 1. Modelo da planta

In [28]:
# Planta do sistema
num = [10]
den = [1, 10, 35, 50, 24]

G = control.TransferFunction(num, den)

### 2. Função de transferência em malha fechada, com controlador PID

In [29]:
def G_MF(kp, kd, ki):

    s = control.TransferFunction.s

    C = kp + ki/s + kd*s

    T = control.feedback(C*G, 1)

    return T

### 3. Função que extrai automaticamente atributos da resposta ao degrau

In [30]:
def stepinfo_EA722(sys, Tmax):

    T = np.arange(0, Tmax, 0.001)

    t, y = control.step_response(sys, T)

    y = np.array(y)

    ypeak = np.max(y)
    ipeak = np.argmax(y)

    tpeak = t[ipeak]

    yhead = y[:ipeak+1]
    ytail = y[ipeak:]

    hf = len(ytail)

    yfinal = np.mean(ytail[int(0.95*hf):])

    # Rise time
    RT_low = 0.1*yfinal
    RT_high = 0.9*yfinal

    ilow = np.argmin(abs(yhead-RT_low))
    ihigh = np.argmin(abs(yhead-RT_high))

    rise_time = t[ihigh] - t[ilow]

    # Overshoot
    overshoot = (ypeak - yfinal)/yfinal

    # Settling time
    ST = 0.02

    settling_time = Tmax

    for i in range(len(ytail)-1,0,-1):
        if abs(ytail[i]-yfinal) > ST*yfinal:
            settling_time = t[ipeak+i]
            break

    S = {
        "RiseTime": rise_time,
        "SettlingTime": settling_time,
        "Overshoot": overshoot,
        "Peak": ypeak,
        "PeakTime": tpeak
    }

    return S

### 4. Função de *fitness* com base nos critérios de desempenho inicialmente definidos

In [31]:
def fitness_PID(kp, kd, ki):

    try:

        sys = G_MF(kp, kd, ki)

        S = stepinfo_EA722(sys,Tmax)

        gm, pm, wcg, wcp = control.margin(sys)

        if np.isnan(S["SettlingTime"]) or np.isnan(S["RiseTime"]) or pm <= 0:
            return 0

        t1 = max(0, S["SettlingTime"] - 3.0)

        t2 = max(0, S["RiseTime"] - 0.6)

        # Eliminação do critério de margem de fase da função-objetivo
        # Restrição de sobressinal <5%,
        t3 = max(0, S["Overshoot"] - 0.05)

        fitness = 1/(t1 + (3.0/0.6)*t2 + t3 + 1)
        return fitness

    except:
        return 0

### 5. Mutação não-uniforme

In [32]:
def mut_nunif(k, y, g):

    r = random.random()

    delta = y*(1-r**((1-k/g)**3))

    return delta

### 6. Parâmetros do algoritmo evolutivo

In [33]:
tam_pop = 50 # @param
pc = 0.5
pm = 0.7

v_inf = 0
v_sup = 20

n_ger = 100 # @param

Tmax = 10 # @param

### 7. Seleção por torneio

In [34]:
# Torneio de 3 indivíduos
def selecao_torneio(pop, fitness):

    candidatos = np.where(fitness>0)[0]

    n_pop = []

    for i in range(len(pop)):

        torneio = np.random.choice(candidatos,3)

        best = torneio[np.argmax(fitness[torneio])]

        n_pop.append(pop[best])

    return np.array(n_pop)

### 8. Crossover

In [35]:
def crossover(pop):

    for j in range(0,len(pop),2):

        if np.random.rand() <= pc:

            if np.random.rand() <= 0.5:

                a = np.random.rand()

                p1 = pop[j]
                p2 = pop[j+1]

                pop[j] = a*p1 + (1-a)*p2
                pop[j+1] = (1-a)*p1 + a*p2

            else:

                for z in range(3):

                    if np.random.rand() <= 0.5:

                        pop[j,z],pop[j+1,z] = pop[j+1,z],pop[j,z]

    return pop

### 9. Mutação

In [36]:
def mutacao(pop, k):

    deltas = []

    n_mut = round(tam_pop*3*pm)

    for _ in range(n_mut):

        linha = random.randint(0, tam_pop-1)
        coluna = random.randint(0, 2)

        if np.random.rand() <= 0.5:

            delta = mut_nunif(k, v_sup-pop[linha,coluna], n_ger)

            pop[linha,coluna] += delta

        else:

            delta = mut_nunif(k, pop[linha,coluna]-v_inf, n_ger)

            pop[linha,coluna] -= delta

        deltas.append(delta)

    return pop, deltas

### 10. Loop evolutivo

In [ ]:
%%time

best_fitness_hist=[]
mean_fitness_hist=[]
best_ind_hist=[]
delta_hist=[]

# população inicial
pop = v_inf + (v_sup-v_inf)*np.random.rand(tam_pop,3)

# avaliação
fitness = np.zeros(tam_pop)

for i in range(tam_pop):
  kp, kd, ki = pop[i]
  fitness[i] = fitness_PID(kp, kd, ki)

best_ind = np.argmax(fitness)
best_fitness = fitness[best_ind]

for g in tqdm(range(n_ger), desc="Evolução", unit="ger"):

    n_pop = selecao_torneio(pop, fitness)

    n_pop = crossover(n_pop)

    n_pop, deltas = mutacao(n_pop, g)

    delta_hist.extend(deltas)

    # avaliação
    fitness = np.zeros(tam_pop)

    for i in range(tam_pop):

        kp, kd, ki = n_pop[i]

        fitness[i] = fitness_PID(kp, kd, ki)

    # best = np.argmax(fitness)

    best_fitness1 = best_fitness
    best_ind1 = best_ind
    best_ind = np.argmax(fitness)
    best_fitness = fitness[best_ind]

    if best_fitness1 > best_fitness:
      worst_ind = np.argmin(fitness)
      n_pop[worst_ind, :] = pop[best_ind1, :]
      fitness[worst_ind] = best_fitness1
      best_fitness = best_fitness1
      best_ind = worst_ind

    best_fitness_hist.append(fitness[best_ind])
    mean_fitness_hist.append(np.mean(fitness))
    best_ind_hist.append(n_pop[best_ind])

    pop = n_pop

Evolução:  83%|████████▎ | 83/100 [08:22<01:42,  6.01s/ger]

In [ ]:
# @title Figura 1: Resposta ao degrau

# parâmetros do melhor indivíduo
kp, kd, ki = pop[best_ind]
best_fitness = fitness_PID(kp, kd, ki)

sys = G_MF(kp, kd, ki)

# parâmetros do Ziegler-Nichols (obtidos pela regra de malha aberta da Exp. 2)
sys2 = G_MF(7.4393,2.8269,4.8943)

t, y = control.step_response(sys, Tmax)
t2, y2 = control.step_response(sys2, Tmax)

# valor final aproximado
y_final = y[-1]

# ---- tempo de subida (10% -> 90%) ----
t10 = t[np.where(y >= 0.1*y_final)[0][0]]
t90 = t[np.where(y >= 0.9*y_final)[0][0]]
t_r = t90 - t10

# ---- tempo de acomodação (banda de 2%) ----
tol = 0.02 * y_final
idx = np.where(np.abs(y - y_final) > tol)[0]

if len(idx) > 0:
    t_s = t[idx[-1] + 1]
else:
    t_s = 0

# ---- margem de fase ----
gm, pm, wg, wp = control.margin(sys)
MF = pm

# ---- overshoot ----
ypeak = np.max(y)
ipeak = np.argmax(y)
tpeak = t[ipeak]

yhead = y[:ipeak+1]
ytail = y[ipeak:]

hf = len(ytail)

yfinal = np.mean(ytail[int(0.95*hf):])

overshoot = (ypeak - yfinal)/yfinal

# =========================
# Prints
# =========================

print(f"kp: {kp:.4f}")
print(f"kd: {kd:.4f}")
print(f"ki: {ki:.4f}")
print(f"Fitness deste melhor indivíduo: {best_fitness:.4f}")
print(f"")
print(f"Tempo de subida (t_r): {t_r:.4f} s")
print(f"Tempo de acomodação (t_s): {t_s:.4f} s")
print(f"Margem de fase (MF): {MF:.2f} graus")
print(f"Overshoot (OS): {100*overshoot:.2f} %")

# =========================
# Plot
# =========================
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=t,
        y=y,
        mode="lines",
        name="Resposta evoluída",
        line=dict(width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=t2,
        y=y2,
        mode="lines",
        name="Resposta Ziegler_Nichols",
        line=dict(width=2)
    )
)

fig.update_layout(
    title=dict(
        text='Resposta ao degrau do melhor controlador',
        x=0.5
    ),
    xaxis_title='Tempo (s)',
    yaxis_title='Saída',
    width=700,
    height=450,
)

fig.show()

kp: 6.2564
kd: 6.0742
ki: 3.6226
Fitness deste melhor indivíduo: 1.0000

Tempo de subida (t_r): 0.5894 s
Tempo de acomodação (t_s): 2.8684 s
Margem de fase (MF): 49.76 graus
Overshoot (OS): 4.50 %


In [ ]:
# @title Figura 2: Histogramas dos ganhos na população da geração final

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("Distribuição de kp", "Distribuição de kd", "Distribuição de ki")
)

fig.add_trace(
    go.Histogram(x=pop[:,0], nbinsx=20, name="kp"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=pop[:,1], nbinsx=20, name="kd"),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(x=pop[:,2], nbinsx=20, name="ki"),
    row=3, col=1
)

fig.update_layout(
    title=dict(
        text='Distribuição dos ganhos do PID na população',
        x=0.5
    ),
    width=700,
    height=700,
    showlegend=False
)

fig.update_xaxes(title_text="kp", row=1, col=1)
fig.update_xaxes(title_text="kd", row=2, col=1)
fig.update_xaxes(title_text="ki", row=3, col=1)

fig.update_yaxes(title_text="Frequência")

fig.show()

In [ ]:
# @title Figura 3: Evolução do fitness
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=best_fitness_hist,
        mode="lines",
        name="Melhor",
        line=dict(color="black", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        y=mean_fitness_hist,
        mode="lines",
        name="Médio",
        line=dict(color="red", width=2)
    )
)

fig.update_layout(
    title=dict(
        text='Fitness ao longo das gerações',
        x=0.5
    ),
    xaxis_title='Geração',
    yaxis_title='Fitness',
    width=700,
    height=450,
)

fig.show()

In [ ]:
# @title Figura 4: Evolução dos ganhos PID
best_ind_hist = np.array(best_ind_hist)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=best_ind_hist[:,0],
        mode="lines",
        name="kp",
        line=dict(color="black", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        y=best_ind_hist[:,1],
        mode="lines",
        name="kd",
        line=dict(color="red", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        y=best_ind_hist[:,2],
        mode="lines",
        name="ki",
        line=dict(color="blue", width=2)
    )
)

fig.update_layout(
    title=dict(
        text='Evolução dos ganhos PID',
        x=0.5
    ),
    xaxis_title='Geração',
    yaxis_title='Valor do ganho',
    width=700,
    height=450,
)

fig.show()

In [ ]:
# @title Figura 5: Intensidade da mutação ao longo das gerações

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=delta_hist,
        mode="lines",
        name="Delta",
        line=dict(width=1)
    )
)

fig.update_layout(
    title=dict(
        text='Intensidade da mutação não-uniforme ao longo das gerações',
        x=0.5
    ),
    xaxis_title='Eventos de mutação',
    yaxis_title='Delta aplicado',
    width=700,
    height=450,
)

fig.show()

## **Não se esqueça de analisar as 5 figuras acima, interpretando os resultados obtidos.**

### 11. Mudança de um dos critérios de desempenho

O enunciado pede:

Remover a margem de fase da função objetivo e incluir:

$$
Overshoot \leq 5\%
$$

Então:

$$
t_3=\begin{cases}
0, & \text{se } OS \leq 0{,}05 \\
OS - 0{,}05, & \text{se } OS > 0{,}05\end{cases}
$$

**Dicas**: Inclua $t_3$ multiplicado por 10, escolha $tam\_pop = 80$ e apresente apenas a Figura 1, após concluir a busca evolutiva.

### 12. Mudança de um dos critérios de desempenho

O enunciado pede uma redução no tempo de subida:

$$
t_r \leq 0{,}5
$$

Então:

$$
t_2=\begin{cases}
0, & \text{se } t_2 \leq 0{,}5 \\
t_2 - 0{,}5, & \text{se } t_2 > 0{,}5\end{cases}
$$

**Dicas**: Inclua $t_3$ multiplicado por 10, acerte o reescalamento do $t_2$, escolha $tam\_pop = 80$ e apresente apenas a Figura 1, após concluir a busca evolutiva.

## (2) Implementação de controle PID por cancelamento de polos

In [ ]:
# @title Parte 2a: Efeito do parâmetro N

s = control.TransferFunction.s

G = 1/(s**3 + 3*s**2 + 3*s + 1)

Kp = 1
Td = 1
Ti = 1

# PID ideal
Gc0 = Kp*(1 + 1/(Ti*s) + Td*s)

# Resposta original
T0 = control.feedback(G*Gc0, 1)

t = np.linspace(0, 20, 1000)
t0, y0 = control.step_response(T0, t)

# ===== Gráfico 1: Resposta ao degrau =====
fig1 = go.Figure()

# Curva original
fig1.add_trace(go.Scatter(x=t0, y=y0, mode='lines', name='Original'))

# Curvas com N
# for N in [1, 10]:
#     Gc = Kp*(1 + 1/(Ti*s) + (Td*s)/(1 + (Td*s)/N))
#     T = control.feedback(G*Gc, 1)
#     t1, y1 = control.step_response(T, t)

#     fig1.add_trace(go.Scatter(
#         x=t1,
#         y=y1,
#         mode='lines',
#         name=f'N={N}'
#     ))

# Valores de N para testar
valores_N = [1, 5, 20, 100]
cores = ['red', 'orange', 'green', 'blue']

for N, cor in zip(valores_N, cores):
    # Controlador com derivada filtrada
    Gc = Kp*(1 + 1/(Ti*s) + (Td*s)/(1 + (Td*s)/N))
    T = control.feedback(G*Gc, 1)
    t1, y1 = control.step_response(T, t)
    
    fig1.add_trace(go.Scatter(
        x=t1, y=y1, 
        mode='lines', 
        name=f'Aproximado N={N}',
        line=dict(color=cor)
    ))

fig1.update_layout(
    title=dict(
        text='Resposta ao degrau',
        x=0.5
    ),
    xaxis_title='Tempo (s)',
    yaxis_title='Saída',
    width=700,
    height=400,
)

fig1.show()

# ===== Gráfico 2: Erro =====
fig2 = go.Figure()

# for N in [1, 10]:
#     Gc = Kp*(1 + 1/(Ti*s) + (Td*s)/(1 + (Td*s)/N))
#     T = control.feedback(G*Gc, 1)
#     t1, y1 = control.step_response(T, t)

#     fig2.add_trace(go.Scatter(
#         x=t,
#         y=y0 - y1,
#         mode='lines',
#         name=f'Erro (N={N})'
#     ))


for N, cor in zip(valores_N, cores):
    Gc = Kp*(1 + 1/(Ti*s) + (Td*s)/(1 + (Td*s)/N))
    T = control.feedback(G*Gc, 1)
    t1, y1 = control.step_response(T, t)
    
    fig2.add_trace(go.Scatter(
        x=t, y=(y0 - y1), 
        mode='lines', 
        name=f'Erro (N={N})',
        line=dict(color=cor)
    ))

fig2.update_layout(
    title=dict(
        text='Erro entre PID ideal e aproximado',
        x=0.5
    ),
    xaxis_title='Tempo (s)',
    yaxis_title='Erro',
    width=700,
    height=400,
)

fig2.show()

Quando aumentamos o valor de N, o polo do filtro passa-baixa ($-\frac{N}{T_D}$) é deslocado para frequências mais altas no plano complexo. Matematicamente, a parcela $\frac{T_D s}{1 + \frac{T_D}{N}s}$ tende a se comportar cada vez mais como uma derivada pura ($T_D s$). Portanto, aumentar o valor de N faz com que a resposta do controlador PID implementável se aproxime da resposta teórica (PID Ideal), reduzindo o erro entre eles

In [ ]:
# @title Parte 2b: Cancelamento de polos

s = control.TransferFunction.s

# -------------------------------
# Planta
# -------------------------------
P = 1/((1+10*s)*(1+s)*(1+0.2*s))

# -------------------------------
# Controladores (Estrutura de Cancelamento)
# -------------------------------
Cp   = 1
Cpi  = (1+10*s)/(10*s)
Cpd  = (1+s)/(1+0.2*s)
Cpid = Cpi*Cpd

# -------------------------------
# Ajuste de ganho (Margem de Fase ≈ 60°)
# -------------------------------
def ajusta_k(C, P):
    L = C*P
    mag, phase, omega = control.bode(L, plot=False)
    phase = np.degrees(phase)
    # Procuramos a fase de -120° para ter margem de 60° (-180 + 60)
    alvo = -120
    idx = np.argmin(np.abs(phase - alvo))
    return 1/mag[idx]

kp   = ajusta_k(Cp, P)
kpi  = ajusta_k(Cpi, P)
kpd  = ajusta_k(Cpd, P)
kpid = ajusta_k(Cpid, P)

# Aplicando os ganhos calculados
Cp   *= kp
Cpi  *= kpi
Cpd  *= kpd
Cpid *= kpid

# -------------------------------
# Malhas Fechadas (Saída T e Esforço U)
# -------------------------------
Lp   = Cp*P
Lpi  = control.minreal(Cpi*P)
Lpd  = control.minreal(Cpd*P)
Lpid = control.minreal(Cpid*P)

Tp   = control.feedback(Lp, 1)
Tpi  = control.feedback(Lpi, 1)
Tpd  = control.feedback(Lpd, 1)
Tpid = control.feedback(Lpid, 1)

Up   = Cp/(1+Lp)
Upi  = Cpi/(1+Lpi)
Upd  = Cpd/(1+Lpd)
Upid = Cpid/(1+Lpid)

# Vetor de Tempo para simulação
t = np.linspace(0, 12, 2000)

# -------------------------------
# GRÁFICO 1: Resposta ao degrau y(t)
# -------------------------------
fig1 = go.Figure()

for T, nome in zip([Tp, Tpi, Tpd, Tpid], ['P','PI','PD','PID']):
    t_out, y = control.step_response(T, t)
    fig1.add_trace(go.Scatter(x=t_out, y=y, mode='lines', name=nome))

fig1.update_layout(
    title=dict(text='Gráfico 3: Resposta ao degrau (saída y(t))', x=0.5),
    xaxis_title='Tempo (s)',
    yaxis_title='Saída y(t)',
    width=700,
    height=400,
)
fig1.show()

# -------------------------------
# GRÁFICO 2: Ação de controle u(t) - P e PI
# -------------------------------
fig2 = go.Figure()

for U, nome in zip([Up, Upi], ['P','PI']):
    t_out, u = control.step_response(U, t)
    fig2.add_trace(go.Scatter(x=t_out, y=u, mode='lines', name=nome))

fig2.update_layout(
    title=dict(text='Gráfico 4: Comportamento das ações de controle (P e PI)', x=0.5),
    xaxis_title='Tempo (s)',
    yaxis_title='Esforço u(t)',
    width=700,
    height=400,
)
fig2.show()

# -------------------------------
# GRÁFICO 3: Ação de controle u(t) - PD e PID
# -------------------------------
fig3 = go.Figure()

for U, nome in zip([Upd, Upid], ['PD','PID']):
    t_out, u = control.step_response(U, t)
    fig3.add_trace(go.Scatter(x=t_out, y=u, mode='lines', name=nome))

fig3.update_layout(
    title=dict(text='Gráfico 5: Comportamento das ações de controle (PD e PID)', x=0.5),
    xaxis_title='Tempo (s)',
    yaxis_title='Esforço u(t)',
    width=700,
    height=400,
)
fig3.show()

# -------------------------------
# CÁLCULO DAS MÉTRICAS PARA A TABELA
# -------------------------------
def metricas(T):
    t_out, y = control.step_response(T, t)
    yss = y[-1]
    ess = 1 - yss
    Mp = (np.max(y) - yss)/yss
    return yss, ess, Mp

def u_max(U):
    t_out, u = control.step_response(U, t)
    return np.max(u)

def tempo_acomodacao(t_out, y, percentual=0.02):
    y_final = y[-1]
    banda = percentual * abs(y_final)
    dentro_banda = np.abs(y - y_final) <= banda
    for i in range(len(y)):
        if np.all(dentro_banda[i:]):
            return t_out[i]
    return t_out[-1]

print("\n========== RESULTADOS PARA A TABELA ==========")

for nome, T, U, k in zip(['P', 'PI', 'PD', 'PID'], [Tp, Tpi, Tpd, Tpid], [Up, Upi, Upd, Upid], [kp, kpi, kpd, kpid]):
    t_out, y = control.step_response(T, t)
    yss, ess, Mp = metricas(T)
    umax = u_max(U)
    ts = tempo_acomodacao(t_out, y)

    print(f"\nControlador {nome}:")
    print(f"k_P: {k:.4f}")
    print(f"M_p: {Mp*100:.2f}%")
    print(f"e(∞): {ess:.4f}")
    print(f"u_max: {umax:.4f}")
    print(f"t_s: {ts:.4f} s")

1 states have been removed from the model
1 states have been removed from the model
2 states have been removed from the model



========== RESULTADOS PARA A TABELA ==========

Controlador P:
k_P: 7.4764
M_p: 14.83%
e(∞): 0.1149
u_max: 7.4764
t_s: 9.3047 s

Controlador PI:
k_P: 5.0696
M_p: 8.67%
e(∞): 0.0063
u_max: 5.1916
t_s: 9.0345 s

Controlador PD:
k_P: 16.6557
M_p: 11.52%
e(∞): 0.0566
u_max: 83.2787
t_s: 2.5393 s

Controlador PID:
k_P: 14.3258
M_p: 7.71%
e(∞): 0.0000
u_max: 71.6290
t_s: 2.8394 s


**Interpretação dos resultados:**

???????????????????????????????

<br>
<br>


|  | $C(s)$ | $C(s)P(s)$ | $k_P$ | $M_p$ | $e(\infty)$ | $u_{\max}$ | $t_s$ |
|:----:|:------:|:----------:|:-----:|:-----:|:-----------:|:----------:|:-----:|
| P | $k_P$ | $\dfrac{k_P}{(1 + 10s)(1 + s)(1 + 0.2s)}$ | 7.4764 | 14.83% | 0.1149 | 7.4764 | 9.3047 s |
| PI | $k_P \dfrac{1 + 10s}{10s}$ | $\dfrac{k_P}{10s(1 + s)(1 + 0.2s)}$ | 5.0696 | 8.67% | 0.0063 | 5.1916 | 9.0345 s |
| PD | $k_P \dfrac{1 + s}{1 + 0.2s}$ | $\dfrac{k_P}{(1 + 10s)(1 + 0.2s)^2}$ | 16.6557 | 11.52% | 0.0566 | 83.2787 | 2.5393 s |
| PID | $k_P \dfrac{(1 + 10s)(1 + s)}{10s(1 + 0.2s)}$ | $\dfrac{k_P}{10s(1 + 0.2s)^2}$ | 14.3258 | 7.71% | 0.0000 | 71.6290 | 2.8394 s |




## (3) Configurações PID alternativas

(a)

> Fazendo $D(s)=N(s)=0$, obtenha as funções  de transferência de malha fechada.



> O que há em comum e em que elas diferem?


In [ ]:
# @title Parte 3b: Efeito da configuração PI&D vs. configuração PID tradicional

# Planta G(s) = 1 / (s^3 + 3 s^2 + 3 s + 1)
num = [1]
den = [1, 3, 3, 1]
G = control.TransferFunction(num, den)

# Parâmetros
Ti = 1
Td = 1
Kp = 1
N = 10
s = control.TransferFunction.s

# Controlador PID com derivada filtrada
Gc = Kp * (1 + 1/(Ti*s) + (Td*s) / (1 + Td*s/N))

# Malha fechada PID
G_c = control.feedback(G*Gc, 1)

# Controlador PI (sem derivada)
Gc1 = Kp * (1 + 1/(Ti*s))

# Função H(s) para realimentação na configuração PI&D
H = ((1 + Kp/N)*Ti*Td*s**2 + Kp*(Ti + Td/N)*s + Kp) / (Kp*(Ti*s + 1)*((Td/N)*s + 1))

# Malha fechada PI&D
G_c1 = control.feedback(G*Gc1, H)

# Plot da resposta ao degrau das duas configurações
t = np.linspace(0, 30, 1000)
t1, y1 = control.step_response(G_c, t)
t2, y2 = control.step_response(G_c1, t)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=t1, y=y1, mode='lines', name='PID'))
fig1.add_trace(go.Scatter(x=t2, y=y2, mode='lines', name='PI&D'))
fig1.update_layout(
    title=dict(
        text='Resposta ao degrau - PID vs. PI&D',
        x=0.5
    ),
    xaxis_title='Tempo (s)',
    yaxis_title='Saída',
    width=700,
    height=400,
)
fig1.show()

# Cálculo das ações de controle

# Lpid = Gc*G em malha aberta simplificada (minimização)
Lpid = control.minreal(Gc*G, tol=1e-4)

# Forma alternativa para ação de controle UPID
Upid = Kp*Gc / (1 + Kp*Gc*G)

# Ação de controle PI&D
Upi_d = (Kp * (1 + 1/(Ti*s))) / (1 + Kp*G*Td*s + Kp*G*(1 + 1/(Ti*s)))

# Resposta temporal das ações de controle
Tmax = 12
t = np.linspace(0, Tmax, 12000)  # mais pontos para maior resolução

t_upid, y_upid = control.step_response(Upid, t)
t_upid_d, y_upi_d = control.step_response(Upi_d, t)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_upid, y=y_upid, mode='lines', name='PID', line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_upid_d, y=y_upi_d, mode='lines', name='PI&D', line=dict(color='blue')))
fig2.update_layout(
    title=dict(
        text='Comportamento das ações de controle no tempo',
        x=0.5
    ),
    xaxis_title='Tempo (s)',
    yaxis_title='Saída',
    width=700,
    height=400,
)
fig2.show()

0 states have been removed from the model


## Interpretação dos resultados


INTERPRETAÇÃO DAS FIGURAS DA PARTE 1

Figura 1 — Resposta ao degrau do melhor controlador

A Figura 1 mostra a resposta ao degrau do melhor controlador encontrado pelo algoritmo evolutivo, em comparação com o controlador obtido pelo método de Ziegler-Nichols. Na execução analisada, o melhor indivíduo apresentou kP = 6,2564, kD = 6,0742 e kI = 3,6226, com fitness igual a 1, o que confirma o atendimento simultâneo das restrições impostas.
Os principais indicadores obtidos também estão dentro das especificações propostas pelo roteiro. Além disso, o sobressinal com percentual de 4,5% representa uma melhora expressiva em relação ao controlador de Ziegler-Nichols.
Dessa forma, a figura evidencia que a resposta evoluída conseguiu encontrar uma solução superior, mais rápida, melhor amortecida e com acomodação melhor do que a usada como referência.

Figura 2 — Distribuição dos ganhos do PID na população final

A Figura 2 apresenta os histogramas dos ganhos kP, kD e kI na última geração da população. Observa-se que os três histogramas estão fortemente concentrados em torno de valores bem definidos, aproximadamente kP ≈ 6,32, kD ≈ 6,02 e kI ≈ 3,62, com dispersões muito pequenas. Essa concentração indica que o algoritmo convergiu de forma bastante consistente para uma região específica do espaço de busca, reduzindo quase totalmente a diversidade genética ao final da evolução. Em termos práticos, isso significa que a maior parte da população final já representa pequenas variações em torno de uma mesma solução ótima ou quase ótima. Esse comportamento confirma que o processo evolutivo não apenas encontrou um bom indivíduo isolado, mas levou praticamente toda a população para uma vizinhança de alto desempenho.

Figura 3 — Evolução do fitness ao longo das gerações

A Figura 3 mostra a evolução do melhor fitness e do fitness médio da população ao longo das 100 gerações. Nota-se que o melhor fitness já começa em um valor relativamente alto, em torno de 0,59, o que indica que a população inicial já continha indivíduos razoavelmente promissores. A partir daí, a curva do melhor fitness cresce em etapas sucessivas, ultrapassando 0,7 por volta da geração 22, chegando a valores acima de 0,8 e 0,9 em torno da geração 50 e atingindo 1 por volta da geração 71. Já o fitness médio cresce mais lentamente, como esperado, pois toda a população demora mais para se adaptar do que apenas o melhor indivíduo. No entanto, a partir das gerações finais, o fitness médio também se aproxima de 1, atingindo valores acima de 0,9 por volta da geração 84 e chegando a 1 no final. Isso mostra que não houve apenas melhora do melhor indivíduo, mas convergência efetiva da população inteira para soluções factíveis e de alto desempenho.

Figura 4 — Evolução dos ganhos PID

A Figura 4 mostra a evolução dos ganhos kP, kD e kI do melhor indivíduo ao longo das gerações. Observa-se que o ganho proporcional diminuiu de aproximadamente 7,90 no início para 6,26 ao final, enquanto o ganho derivativo aumentou de cerca de 5,86 para 6,07. Já o ganho integral sofreu redução mais significativa, saindo de aproximadamente 4,61 para 3,62, embora tenha apresentado oscilações intermediárias mais acentuadas ao longo do processo. Esse comportamento indica que o algoritmo foi ajustando gradualmente o balanço entre rapidez e amortecimento da resposta: a redução de kP e kI, acompanhada de uma leve elevação de kD, sugere que a solução final passou a privilegiar maior amortecimento e menor tendência ao sobressinal, sem perder a rapidez exigida pelas restrições. Em outras palavras, a figura mostra que a sintonia evolutiva refinou a ação derivativa e reduziu o excesso de agressividade proporcional e integral até atingir uma solução mais equilibrada.

Figura 5 — Intensidade da mutação não uniforme ao longo das gerações

A Figura 5 apresenta a intensidade das mutações aplicadas ao longo da execução do algoritmo. Nota-se claramente que os valores de Δ são altos no início e diminuem progressivamente até se tornarem praticamente nulos no final. Na execução analisada, as mutações iniciais chegaram a valores máximos da ordem de 18, com média dos primeiros eventos em torno de 5,36, enquanto nas mutações finais a média caiu para aproximadamente 10^-5. Esse comportamento confirma que a mutação não uniforme foi implementada corretamente: no início da busca, ela promove grandes perturbações para explorar diferentes regiões do espaço de busca; nas últimas gerações, as perturbações se tornam extremamente pequenas, atuando apenas no refinamento fino da melhor solução encontrada. Assim, a figura evidencia a transição típica entre uma fase inicial de exploração global e uma fase final de explotação local.

Fechamento da análise das 5 figuras

De forma geral, as cinco figuras mostram um processo evolutivo muito bem-sucedido. A resposta ao degrau final atende completamente às restrições especificadas, apresentando inclusive sobressinal baixo; a população final está fortemente concentrada em torno de uma única região do espaço de busca; o fitness do melhor indivíduo e o fitness médio convergem para 1; os ganhos evoluem até uma configuração mais equilibrada; e a mutação perde intensidade ao longo das gerações, evidenciando refinamento progressivo. Em conjunto, os gráficos confirmam que o algoritmo evolutivo conseguiu não apenas encontrar uma solução factível, mas conduzir toda a população a uma região de ótimo desempenho para o problema proposto.